# Kafka Producer — 按事件时间回放 ODS 黄车数据

从 **ODS 层**（未清洗）读取 `REPLAY_DATE` 当天 `[REPLAY_START, REPLAY_END)` 时段的黄车记录，按上车时间排序后逐条写入 Kafka topic `tlc-trips`。

- 读 ODS 而非 DWD：实时端必须**自己清洗**，与离线 DWD 对账才有意义
- 用 Spark 读取并用 `date_format` 格式化时间：与 03/09 使用同一时间解析方式，避免 pandas/Spark 时区处理不一致
- 按完整时间段回放（而不是前 N 条）：对账窗口边界整齐
- 最后发送一条 **watermark heartbeat**（事件时间 = END + 10 分钟，虚拟区域 `-1`）：推进水位线，让最后几个窗口也能关闭输出；消费端输出时排除区域 `-1`

运行前：先执行 `bash scripts/reset_stream.sh`（重建 topic、清空输出与 checkpoint）。

In [ ]:
import os, json, time
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, functions as F
from kafka import KafkaProducer, KafkaConsumer

BASE          = os.path.expanduser("~/tlc-spark-pipeline")
BOOTSTRAP     = "localhost:9092"
TOPIC         = "tlc-trips"
REPLAY_START  = os.environ.get("REPLAY_START", "2024-03-15 08:00:00")
REPLAY_END    = os.environ.get("REPLAY_END",   "2024-03-15 09:00:00")
DELAY_SECONDS = float(os.environ.get("REPLAY_DELAY", "0.01"))   # 每条间隔，模拟实时到达

d = datetime.strptime(REPLAY_START, "%Y-%m-%d %H:%M:%S")
ODS_PATH = f"{BASE}/data/warehouse/ods/yellow/pickup_year={d.year}/pickup_month={d.month}/pickup_day={d.day}"
print(f"Replay window: [{REPLAY_START}, {REPLAY_END})  from {ODS_PATH}")

## 1. 用 Spark 读取 ODS 并格式化

In [ ]:
spark = (SparkSession.builder.appName("TLC-Kafka-Producer")
         .config("spark.driver.memory", "2g").getOrCreate())
spark.sparkContext.setLogLevel("WARN")

TS = "yyyy-MM-dd HH:mm:ss"
replay = (
    spark.read.parquet(ODS_PATH)
    .filter((F.col("pickup_datetime") >= F.lit(REPLAY_START).cast("timestamp")) &
            (F.col("pickup_datetime") <  F.lit(REPLAY_END).cast("timestamp")))
    .select(
        F.col("VendorID").cast("int").alias("VendorID"),
        F.date_format("pickup_datetime",  TS).alias("pickup_datetime"),
        F.date_format("dropoff_datetime", TS).alias("dropoff_datetime"),
        F.col("PULocationID").cast("int").alias("PULocationID"),
        F.col("DOLocationID").cast("int").alias("DOLocationID"),
        F.col("passenger_count").cast("int").alias("passenger_count"),
        F.col("trip_distance").cast("double").alias("trip_distance"),
        F.col("fare_amount").cast("double").alias("fare_amount"),
        F.col("tip_amount").cast("double").alias("tip_amount"),
        F.col("total_amount").cast("double").alias("total_amount"),
        F.col("payment_type").cast("int").alias("payment_type"),
        F.lit("yellow").alias("source_type"),
    )
    .orderBy("pickup_datetime")
)
rows = [r.asDict() for r in replay.collect()]
spark.stop()

print(f"Records to replay: {len(rows):,}")
print(f"First: {rows[0]['pickup_datetime']}   Last: {rows[-1]['pickup_datetime']}")

## 2. 发送到 Kafka（按 PULocationID 分区）

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    acks="all",
    key_serializer=lambda k: str(k).encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

errors = []
t0 = time.time()
for i, rec in enumerate(rows, 1):
    rec["is_heartbeat"] = False
    producer.send(TOPIC, key=rec["PULocationID"], value=rec).add_errback(errors.append)
    if i % 1000 == 0:
        print(f"sent {i:,}  event_time={rec['pickup_datetime']}")
    time.sleep(DELAY_SECONDS)
producer.flush()

# watermark heartbeat：事件时间在回放窗口之后，推进水位线以关闭最后的窗口
hb_time = (datetime.strptime(REPLAY_END, "%Y-%m-%d %H:%M:%S") + timedelta(minutes=10)).strftime("%Y-%m-%d %H:%M:%S")
hb_drop = (datetime.strptime(hb_time, "%Y-%m-%d %H:%M:%S") + timedelta(minutes=5)).strftime("%Y-%m-%d %H:%M:%S")
# heartbeat 设计成一条能通过全部清洗规则的记录，放在虚拟区域 -1；消费端输出时排除该区域
producer.send(TOPIC, key="heartbeat", value={
    "is_heartbeat": True, "pickup_datetime": hb_time, "dropoff_datetime": hb_drop,
    "PULocationID": -1, "DOLocationID": -1, "trip_distance": 1.0, "fare_amount": 1.0,
    "source_type": "heartbeat"})
producer.flush()
producer.close()

print(f"\nSent {len(rows):,} records + 1 heartbeat in {time.time()-t0:.0f}s, errors={len(errors)}")
assert not errors, errors[:3]

## 3. 校验：从头读回 topic，条数应等于发送数 + 1

In [ ]:
consumer = KafkaConsumer(
    TOPIC, bootstrap_servers=BOOTSTRAP,
    auto_offset_reset="earliest", enable_auto_commit=False,
    group_id=None, consumer_timeout_ms=5000,
)
n, parts = 0, set()
for msg in consumer:
    n += 1
    parts.add(msg.partition)
consumer.close()

print(f"Messages in topic: {n:,}  (expected {len(rows)+1:,})  partitions used: {sorted(parts)}")
assert n == len(rows) + 1, "topic 里有残留或丢失的消息 — 先运行 scripts/reset_stream.sh"